In [4]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import pandas as pd
from pathlib import Path
import re
# ============================================================
# CONFIGURATION
# ============================================================
BASE_URL = "https://books.toscrape.com"
FIXED_GBP_TO_INR = 105.50
DATA_DIR = Path(".").resolve()
DB_PATH = DATA_DIR / "books.db"
SQL_OUTPUT_PATH = DATA_DIR / "sql_outputs.txt"
MERGE_OUTPUT_PATH = DATA_DIR / "merge_comparison.csv"
# Three categories are enough because each category contains
# approximately 20 books, giving us at least 60 records.
TARGET_CATEGORIES = [
    "travel_2",
    "mystery_3",
    "historical-fiction_4"
]
# ============================================================
# HTTP SESSION
# ============================================================
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/151.0 Safari/537.36"
    )
})
# ============================================================
# HELPER FUNCTIONS
# ============================================================
def get_soup(url):
    """
    Download a webpage and return BeautifulSoup object.
    Raises an error for unsuccessful HTTP responses.
    """
    response = session.get(url, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")
def parse_price(price_text):
    """
    Convert £51.77 into 51.77.
    """
    if not price_text:
        raise ValueError("Missing price")
    cleaned = price_text.replace("£", "").replace("Â", "").strip()
    return float(cleaned)
def parse_rating(rating_element):
    """
    Convert:
        One -> 1
        Two -> 2
        Three -> 3
        Four -> 4
        Five -> 5
    """
    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }
    if rating_element is None:
        raise ValueError("Missing rating")
    classes = rating_element.get("class", [])
    rating_word = next(
        (value for value in classes if value in rating_map),
        None
    )
    if rating_word is None:
        raise ValueError("Unexpected rating")
    return rating_map[rating_word]
def parse_availability(availability_text):
    """
    Convert:
        'In stock (22 available)' -> True
        'Out of stock' -> False
    """
    if not availability_text:
        raise ValueError("Missing availability")
    text = availability_text.strip().lower()
    if "in stock" in text:
        return True
    if "out of stock" in text:
        return False
    raise ValueError("Unexpected availability text")
# ============================================================
# SCRAPING
# ============================================================
def scrape_category(category_slug):
    """
    Scrape every page belonging to one category.
    """
    books = []
    page_number = 1
    while True:
        if page_number == 1:
            url = (
                f"{BASE_URL}/catalogue/category/books/"
                f"{category_slug}/index.html"
            )
        else:
            url = (
                f"{BASE_URL}/catalogue/category/books/"
                f"{category_slug}/page-{page_number}.html"
            )
        print(f"Scraping: {url}")
        try:
            soup = get_soup(url)
        except requests.HTTPError:
            # No more pages
            break
        product_list = soup.select("article.product_pod")
        if not product_list:
            break
        for product in product_list:
            try:
                title_element = product.select_one("h3 a")
                price_element = product.select_one(".price_color")
                rating_element = product.select_one(".star-rating")
                availability_element = product.select_one(
                    ".availability"
                )
                title = title_element.get("title", "").strip()
                price_gbp = parse_price(
                    price_element.get_text(strip=True)
                )
                rating = parse_rating(rating_element)
                availability_text = (
                    availability_element.get_text(" ", strip=True)
                )
                in_stock = parse_availability(
                    availability_text
                )
                books.append({
                    "title": title,
                    "price_gbp": price_gbp,
                    "star_rating": (
                        rating_element.get("class", [])[-1]
                    ),
                    "rating": rating,
                    "availability": availability_text,
                    "in_stock": in_stock,
                    "category": category_slug.replace("_2", "")
                })
            except (ValueError, AttributeError, TypeError) as error:
                # Assignment allows dropping rows where parsing fails.
                print(
                    f"Skipping malformed row: {error}"
                )
        # Check whether another page exists
        next_button = soup.select_one("li.next a")
        if next_button is None:
            break
        page_number += 1
    return books
def scrape_all_categories():
    """
    Scrape all target categories.
    """
    all_books = []
    for category in TARGET_CATEGORIES:
        category_books = scrape_category(category)
        print(
            f"{category}: {len(category_books)} books collected"
        )
        all_books.extend(category_books)
    return pd.DataFrame(all_books)
# ============================================================
# CLEANING AND TRANSFORMATION
# ============================================================
def clean_data(df):
    """
    Clean data types and calculate INR prices.
    """
    df = df.copy()
    # Remove duplicate titles
    df = df.drop_duplicates(
        subset=["title", "category"]
    )
    # Ensure price is numeric
    df["price_gbp"] = pd.to_numeric(
        df["price_gbp"],
        errors="coerce"
    )
    # Ensure rating is numeric
    df["rating"] = pd.to_numeric(
        df["rating"],
        errors="coerce"
    )
    # Ensure boolean
    df["in_stock"] = df["in_stock"].astype(bool)
    # Drop rows where required numeric fields failed.
    # This is preferable here to inventing a price/rating for
    # a malformed scraped product.
    before = len(df)
    df = df.dropna(
        subset=["price_gbp", "rating"]
    )
    removed = before - len(df)
    print(
        f"Removed {removed} malformed rows."
    )
   # Rating should be 1-5
    df = df[
        df["rating"].between(1, 5)
    ]
    df["rating"] = df["rating"].astype(int)
    # Required fixed project conversion rate
    df["price_inr"] = (
        df["price_gbp"] * FIXED_GBP_TO_INR
    ).round(2)
    return df
# ============================================================
# SQLITE DATABASE
# ============================================================
def create_database(df):
    """
    Create normalized SQLite database.
    """
    if DB_PATH.exists():
        DB_PATH.unlink()
    connection = sqlite3.connect(DB_PATH)
    cursor = connection.cursor()
    # Enable foreign key constraints
    cursor.execute(
        "PRAGMA foreign_keys = ON"
    )
    # Categories table
    cursor.execute("""
        CREATE TABLE categories (
            category_id INTEGER PRIMARY KEY AUTOINCREMENT,
            category_name TEXT NOT NULL UNIQUE
        )
    """)
    # Books table
    cursor.execute("""
        CREATE TABLE books (
            book_id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            price_gbp REAL NOT NULL,
            price_inr REAL NOT NULL,
            rating INTEGER NOT NULL,
            in_stock INTEGER NOT NULL,
            category_id INTEGER NOT NULL,
            FOREIGN KEY (category_id)
                REFERENCES categories(category_id)
        )
    """)
    # Insert categories
    categories = sorted(
        df["category"].unique()
    )
    cursor.executemany(
        """
        INSERT INTO categories(category_name)
        VALUES (?)
        """,
        [(category,) for category in categories]
    )
    # Retrieve category IDs
    category_map = dict(
        cursor.execute(
            "SELECT category_name, category_id FROM categories"
        ).fetchall()
    )
    # Insert books
    book_rows = []
    for _, row in df.iterrows():
        book_rows.append((
            row["title"],
            float(row["price_gbp"]),
            float(row["price_inr"]),
            int(row["rating"]),
            int(row["in_stock"]),
            category_map[row["category"]]
        ))
    cursor.executemany(
        """
        INSERT INTO books (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        book_rows
    )
    connection.commit()
    return connection
# ============================================================
# SQL QUERIES
# ============================================================
QUERIES = {
    "1_SELECT_WHERE": """
        SELECT title, price_gbp, rating
        FROM books
        WHERE rating >= 4
        ORDER BY rating DESC;
    """,
    "2_ORDER_BY_LIMIT": """
        SELECT title, price_gbp, price_inr
        FROM books
        ORDER BY price_gbp DESC
        LIMIT 10;
    """,
    "3_DISTINCT": """
        SELECT DISTINCT rating
        FROM books
        ORDER BY rating;
    """,
    "4_BETWEEN": """
        SELECT title, price_gbp, rating
        FROM books
        WHERE price_gbp BETWEEN 20 AND 40
        ORDER BY price_gbp;
    """,
    "5_IN": """
        SELECT title, rating, category_id
        FROM books
        WHERE rating IN (4, 5)
        ORDER BY rating DESC, title;
    """,
    "6_JOIN": """
        SELECT
            c.category_name,
            b.title,
            b.rating,
            b.price_gbp,
            b.price_inr,
            b.in_stock
        FROM books b
        JOIN categories c
            ON b.category_id = c.category_id
        ORDER BY
            c.category_name,
            b.rating DESC,
            b.price_gbp DESC;
    """
}
def execute_queries(connection):
    """
    Execute SQL queries and save their outputs.
    """
    outputs = {}
    with open(SQL_OUTPUT_PATH, "w", encoding="utf-8") as file:
        for query_name, query in QUERIES.items():
            print("\n" + "=" * 70)
            print(query_name)
            print("=" * 70)
            print(query)
            file.write("\n" + "=" * 70 + "\n")
            file.write(query_name + "\n")
            file.write("=" * 70 + "\n")
            file.write(query + "\n")
            result = pd.read_sql(
                query,
                connection
            )
            outputs[query_name] = result
            print(result.to_string(index=False))
            file.write(
                result.to_string(index=False)
            )
            file.write("\n\n")
    return outputs
# ============================================================
# PANDAS VALIDATION
# ============================================================
def pandas_join_comparison(connection, cleaned_df):
    """
    Reproduce the SQL JOIN using pd.merge().
    """
    # Read the two normalized tables
    books_df = pd.read_sql(
        "SELECT * FROM books",
        connection
    )
    categories_df = pd.read_sql(
        "SELECT * FROM categories",
        connection
    )
    # SQL JOIN equivalent using pandas
    merged_df = pd.merge(
        books_df,
        categories_df,
        on="category_id",
        how="inner"
    )
    pandas_join = merged_df[
        [
            "category_name",
            "title",
            "rating",
            "price_gbp",
            "price_inr",
            "in_stock"
        ]
    ].sort_values(
        by=[
            "category_name",
            "rating",
            "price_gbp"
        ],
        ascending=[
            True,
            False,
            False
        ]
    ).reset_index(drop=True)
    # SQL JOIN result
    sql_join = pd.read_sql(
        QUERIES["6_JOIN"],
        connection
    ).reset_index(drop=True)
    # Ensure identical data types for comparison
    pandas_join["in_stock"] = (
        pandas_join["in_stock"].astype(int)
    )
    sql_join["in_stock"] = (
        sql_join["in_stock"].astype(int)
    )
    pandas_join["price_gbp"] = (
        pandas_join["price_gbp"].round(2)
    )
    pandas_join["price_inr"] = (
        pandas_join["price_inr"].round(2)
    )
    sql_join["price_gbp"] = (
        sql_join["price_gbp"].round(2)
    )
    sql_join["price_inr"] = (
        sql_join["price_inr"].round(2)
    )
    # Verify equivalence
    equivalent = pandas_join.equals(sql_join)
    print("\n" + "=" * 70)
    print("SQL JOIN vs pandas.merge()")
    print("=" * 70)
    print(
        f"Results equivalent: {equivalent}"
    )
    print("\nSQL JOIN result:")
    print(sql_join.head(10).to_string(index=False))
    print("\npandas.merge() result:")
    print(pandas_join.head(10).to_string(index=False))
    # Save side-by-side comparison
    comparison = pd.concat(
        [
            sql_join.add_prefix("sql_"),
            pandas_join.add_prefix("pandas_")
        ],
        axis=1
    )
    comparison.to_csv(
        MERGE_OUTPUT_PATH,
        index=False
    )
    return equivalent
# ============================================================
# MAIN PIPELINE
# ============================================================
def main():
    print("=" * 70)
    print("ZEpto DATA PIPELINE")
    print("=" * 70)
    # --------------------------------------------------------
    # STEP 1: SCRAPE
    # --------------------------------------------------------
    print("\nSTEP 1: Scraping books...")
    raw_df = scrape_all_categories()
    print(
        f"\nTotal scraped rows: {len(raw_df)}"
    )
    print(
        f"Categories: {raw_df['category'].nunique()}"
    )
    if len(raw_df) < 60:
        raise RuntimeError(
            "Dataset contains fewer than 60 books."
        )
    if raw_df["category"].nunique() < 3:
        raise RuntimeError(
            "Dataset contains fewer than 3 categories."
        )
    # --------------------------------------------------------
    # STEP 2: CLEAN
    # --------------------------------------------------------
    print("\nSTEP 2: Cleaning data...")
    cleaned_df = clean_data(raw_df)
    # --------------------------------------------------------
    # STEP 3: DATABASE
    # --------------------------------------------------------
    print("\nSTEP 3: Creating SQLite database...")
    connection = create_database(
        cleaned_df
    )
    # --------------------------------------------------------
    # STEP 4: SQL
    # --------------------------------------------------------
    print("\nSTEP 4: Executing SQL queries...")
    execute_queries(connection)
    # --------------------------------------------------------
    # STEP 5: PANDAS VALIDATION
    # --------------------------------------------------------
    print(
        "\nSTEP 5: Comparing SQL JOIN with pandas.merge()..."
    )
    equivalent = pandas_join_comparison(
        connection,
        cleaned_df
    )
    if not equivalent:
        raise RuntimeError(
            "SQL JOIN and pandas.merge() results do not match."
        )
    # --------------------------------------------------------
    # FINAL VALIDATION
    # --------------------------------------------------------
    book_count = pd.read_sql(
        "SELECT COUNT(*) AS count FROM books",
        connection
    ).iloc[0]["count"]
    category_count = pd.read_sql(
        "SELECT COUNT(*) AS count FROM categories",
        connection
    ).iloc[0]["count"]
    print("\n" + "=" * 70)
    print("PIPELINE COMPLETE")
    print("=" * 70)
    print(f"Books inserted     : {book_count}")
    print(f"Categories inserted: {category_count}")
    print(f"Conversion rate    : 1 GBP = {FIXED_GBP_TO_INR} INR")
    print(f"Database           : {DB_PATH}")
    print(f"SQL outputs        : {SQL_OUTPUT_PATH}")
    print(f"Merge comparison   : {MERGE_OUTPUT_PATH}")
    connection.close()
if __name__ == "__main__":
    main()

ZEpto DATA PIPELINE

STEP 1: Scraping books...
Scraping: https://books.toscrape.com/catalogue/category/books/travel_2/index.html
travel_2: 11 books collected
Scraping: https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Scraping: https://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html
mystery_3: 32 books collected
Scraping: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Scraping: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/page-2.html
historical-fiction_4: 26 books collected

Total scraped rows: 69
Categories: 3

STEP 2: Cleaning data...
Removed 0 malformed rows.

STEP 3: Creating SQLite database...

STEP 4: Executing SQL queries...

1_SELECT_WHERE

        SELECT title, price_gbp, rating
        FROM books
        WHERE rating >= 4
        ORDER BY rating DESC;
    
                                                                   title  price_gbp  rating
                         